<a href="https://colab.research.google.com/github/AdyashaGiri/Salesforcasting-simple-ML-project-/blob/main/SalesForcasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -Uq upgini catboost

In [ ]:
from os.path import exists

In [ ]:
import pandas as pd
from os.path import exists

df_path = "train.csv.zip" if exists("train.csv.zip") else "https://github.com/upgini/upgini/raw/main/notebooks/train.csv.zip"
df = pd.read_csv(df_path)
df = df.sample(n=19_000, random_state=0)
df["store"] = df["store"].astype(str)
df["item"] = df["item"].astype(str)

df["date"] = pd.to_datetime(df["date"])

df.sort_values("date", inplace=True)
df.reset_index(inplace=True, drop=True)
df.head()



,date,store,item,sales
0,2013-01-01,7,5,5
1,2013-01-01,4,9,19
2,2013-01-01,1,33,37
3,2013-01-01,3,41,14
4,2013-01-01,5,24,26


In [ ]:
train = df[df["date"]< "2017-01-01"]
test= df[df["date"]>= "2017-01-01"]

In [ ]:
train_features = train.drop(columns=["sales"])
train_target = train["sales"]
test_features = test.drop(columns=["sales"])
test_target = test["sales"]

In [ ]:
from upgini import FeaturesEnricher, SearchKey
from upgini.metadata import CVType

# 1. Fixed the argument name to 'cv' and corrected 'time_series'
enricher = FeaturesEnricher(
    search_keys={
        "date": SearchKey.DATE,
    },
    cv=CVType.time_series
)

# 2. Fixed the eval_set syntax to correctly pass a list of tuples
enricher.fit(
    train_features,
    train_target,
    eval_set=[(test_features, test_target)]
)

[============================================================] 100% Finished

<IPython.core.display.Javascript object>

WARNING #1: Search started with DATE search key only
Try to add other keys like the COUNTRY, POSTAL_CODE, PHONE NUMBER, EMAIL/HEM, IP to your training dataset
for search through all the available data sources.
See docs https://github.com/upgini/upgini#-total-239-countries-and-up-to-41-years-of-history

Detected task type: ModelTaskType.REGRESSION. Reason: date search key is present, treating as regression
You can set task type manually with argument `model_task_type` of FeaturesEnricher constructor if task type detected incorrectly

WARNING #2: Your training sample is unstable in number of rows per date. It is recommended to redesign the training sample



Column name,Status,Errors
date,All valid,-
target,All valid,-


<IPython.core.display.Javascript object>

...

f_events_date_week_sin1_847b5db1,12.3407,,100.0000,"0.0, -0.4339, -0.9749",Upgini,Calendar data,Daily
f_economic_date_cpi_umap_1_ba1c4045,8.0704,0.0000,100.0000,"0.7745, 7.2516, 0.7891",Upgini,World economic indicators,Daily
f_events_date_year_cos1_9014a856,4.9298,0.0000,100.0000,"0.3253, -0.263, -0.3496",Upgini,Calendar data,Daily
f_autofe_roll_2d_norm_mean_b3210883b2,3.9643,,85.6719,"0.438, -0.8822, 4.5948","Training dataset,Upgini","AutoFE: features from Training dataset,Markets data",Daily
f_financial_date_crude_oil_7d_to_1y_c3e0ad17,2.8119,0.0000,100.0000,"1.0001, 1.0769, 1.0154",Upgini,Markets data,Daily
f_financial_date_silver_gap_f5b63f5b,1.9692,0.0000,100.0000,"-0.105, 0.491, -0.478",Upgini,Markets data,Daily
item,0.7881,,100.0000,"40, 30, 18",,,
store,0.3098,,100.0000,"5, 10, 7",,,


Upgini,Calendar data,17.2705,2
Upgini,World economic indicators,8.0704,1
Upgini,Markets data,4.7811,2
"Training dataset,Upgini","AutoFE: features from Training dataset,Markets data",3.9643,1


"Training dataset,Markets data",f_autofe_roll_2d_norm_mean_b3210883b2,f_financial_date_usd_eur_gap_c8eb8d4a,roll_2d_norm_mean


We detected 48 outliers in your sample.
Examples of outliers with maximum value of target:
33    205
17    196
12    187
Name: target, dtype: int64
Outliers will be excluded during the metrics calculation.
Calculating accuracy uplift after enrichment...
y distributions from the training sample and eval_set differ according to the Kolmogorov-Smirnov test,
which makes metrics between the train and eval_set incomparable.


Train,9418,53.3352,0.324 ± 0.109,0.304 ± 0.098,0.0200,6.2%
Eval 1,3764,58.5994,0.278 ± 0.009,0.271 ± 0.024,0.0060,2.3%


In [ ]:
from catboost import CatBoostRegressor
from catboost.utils import eval_metric

model = CatBoostRegressor(verbose=False, allow_writing_files=False, random_state=0)

enricher.calculate_metrics(
    train_features, train_target,
    eval_set = [(test_features, test_target)],
    estimator = model,
    scoring = "mean_absolute_percentage_error"
)

Calculating accuracy uplift after enrichment...
y distributions from the training sample and eval_set differ according to the Kolmogorov-Smirnov test,
which makes metrics between the train and eval_set incomparable.


,Dataset type,Rows,Mean target,Baseline MAPE,Enriched MAPE,"Uplift, abs","Uplift, %"
0,Train,9418,53.3352,0.288 ± 0.096,0.219 ± 0.092,0.069,23.8%
1,Eval 1,3764,58.5994,0.247 ± 0.008,0.212 ± 0.021,0.035,14.3%


In [ ]:
# 1. Take exactly what's left in your daily limit (906 rows)
train_features_subset = train_features.iloc[:900]
test_features_subset = test_features.iloc[:900]

# 2. Transform them and assign them to the variables
enriched_train_features = enricher.transform(train_features_subset, keep_input=True)
enriched_test_features = enricher.transform(test_features_subset, keep_input=True)

# 3. VERIFY THEY ARE NOT NONE
print("Train features type:", type(enriched_train_features))
print("Test features type:", type(enriched_test_features))

[============================================================] 100% Finished

You use Trial access to Upgini data enrichment. Limit for Trial: 1000 rows. You have already enriched: 94 rows.
WARNING #1: Search started with DATE search key only
Try to add other keys like the COUNTRY, POSTAL_CODE, PHONE NUMBER, EMAIL/HEM, IP to your training dataset
for search through all the available data sources.
See docs https://github.com/upgini/upgini#-total-239-countries-and-up-to-41-years-of-history



Column name,Status,Errors
date,All valid,-




Running transform request, id=cd8f0d74-c648-4756-9dcc-01e265862b23
We'll send email notification once it's completed, just use your personal api_key from profile.upgini.com

Retrieving selected features from data sources...


[============================================================] 100% Finished

Unregistered-user limit: 822 rows remaining; you requested 900.


Button(description='Get an API KEY', layout=Layout(width='auto'), style=ButtonStyle(), tooltip='Register', _do…

Train features type: <class 'pandas.core.frame.DataFrame'>
Test features type: <class 'NoneType'>


In [ ]:
model.fit(train_features, train_target)
preds = model.predict(test_features)
eval_metric(test_target.values, preds, "SMAPE")

[37.65141857448004]

In [ ]:
# Update targets to match the 906 rows
train_target_subset = train_target.iloc[:900]
test_target_subset = test_target.iloc[:900]

# Fit and predict
model.fit(enriched_train_features, train_target_subset)
enriched_preds = model.predict(enriched_test_features)

# Evaluate
eval_metric(test_target_subset.values, enriched_preds, "SMAPE")

CatBoostError: Invalid data type=<class 'NoneType'> : must be list, numpy.ndarray, pandas.Series, pandas.DataFrame, polars.Series, polars.DataFrame scipy.sparse matrix, catboost.FeaturesData or catboost.Pool